In [7]:
# Import libraries
import pandas as pd
import numpy as np
import joblib, json
import warnings
warnings.filterwarnings('ignore')
from xgboost import XGBClassifier

# Add project root to path
import os
import sys
sys.path.append(os.path.abspath('../'))
from src.data.loader import load_csv
from src.models.trainer import split_data, encoding, class_weights, clean_feature_names

# Settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Libraries imported successfully!")
print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully!
Python version: 3.13.5
Pandas version: 2.3.1
NumPy version: 2.3.2


In [8]:
# Load data
train = load_csv('../data/external/train_top_features.csv')
print()
test = load_csv('../data/external/test_top_features.csv')

👉 Loading: ../data/external/train_top_features.csv
DATASET INFORMATION
Shape: 1,296,675 rows × 10 columns
Memory usage: 150.87 MB

Column types:
int64      6
float64    3
object     1
Name: count, dtype: int64

Missing values: 0

👉 Loading: ../data/external/test_top_features.csv
DATASET INFORMATION
Shape: 555,719 rows × 10 columns
Memory usage: 64.66 MB

Column types:
int64      6
float64    3
object     1
Name: count, dtype: int64

Missing values: 0


In [9]:
X_train, y_train = split_data(train)
X_test,  y_test  = split_data(test)

X_train = clean_feature_names(X_train)
X_test  = clean_feature_names(X_test)

X_train, X_test = encoding(X_train, X_test)
scale_pos_weight = class_weights(y_train)

Feature shape: (1296675, 9)
scale_pos_weight: 171.75


In [10]:
# Save model
best_params = {
    'n_estimators'      : 300,
    'max_depth'         : 10,
    'learning_rate'     : 0.024,
    'subsample'         : 0.97,
    'colsample_bytree'  : 0.94,
    'min_child_weight'  : 15,
    'reg_alpha'         : 0.2,
    'reg_lambda'        : 0.0007,
    'gamma'             : 4.8,
    'scale_pos_weight'  : scale_pos_weight,
    'random_state'      : 42,
    'n_jobs'            : -1,
    'eval_metric'       : 'aucpr'
}
xgb_tuned = XGBClassifier(**best_params)
xgb_tuned.fit(X_train, y_train)

joblib.dump(xgb_tuned, '../models/xgb_fraud_final.pkl')

# Save metadata
FEATURES = [
    'amt', 'category_fraud_rate', 'category_freq', 'city_pop',
    'gender', 'hour', 'age', 'merchant_fraud_rate', 'merchant_freq',
]
metadata = {
    'features'  : FEATURES,
    'threshold' : 0.9534,
    'metrics'   : {
        'pr_auc'   : 0.8975,
        'roc_auc'  : 0.9978,
        'precision': 0.762,
        'recall'   : 0.850,
        'f2'       : 0.8311,
    },
    'model_params': best_params,
}
with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Model saved.")

Model saved.


In [11]:
# Inference function
def predict_fraud(transactions: pd.DataFrame,
                  model, features: list, threshold: float) -> pd.DataFrame:
    """
    Parameters
    ----------
    transactions : DataFrame with at minimum the 9 model features
    model        : fitted XGBClassifier
    features     : ordered feature list from metadata
    threshold    : decision threshold from metadata

    Returns
    -------
    DataFrame with columns: fraud_probability, is_fraud_predicted
    """
    X = transactions[features]
    probs = model.predict_proba(X)[:, 1]
    return pd.DataFrame({
        'fraud_probability'  : probs,
        'is_fraud_predicted' : (probs >= threshold).astype(int),
    }, index=transactions.index)


# Quick smoke test
sample = X_test.tail(10)
result = predict_fraud(sample, xgb_tuned, FEATURES, threshold=0.9534)
print(result)

        fraud_probability  is_fraud_predicted
555709           0.003947                   0
555710           0.000275                   0
555711           0.002368                   0
555712           0.170716                   0
555713           0.002340                   0
555714           0.002628                   0
555715           0.000747                   0
555716           0.002398                   0
555717           0.625778                   0
555718           0.001303                   0
